### TinyStories: генерация коротких текстов

В этом задании вы реализуете transformer decoder и обучите его на датасете TinyStories, а также продемонстрируете его работу, сгенерировав несколько примеров текстов по заданному началу.

Большая часть блоков уже описана в слайдах и в ноутбуках [notebooks/transformer_attention.ipynb](../notebooks/transformer_attention.ipynb) и [notebooks/transformer_model.ipynb](../notebooks/transformer_model.ipynb), вам осталось только собрать их в модуль `TransformerDecoder`.

In [1]:
import torch
import torch.nn.functional as F
from datasets import load_dataset
from torch import nn, Tensor
from torch.utils.data import DataLoader
from transformers import GPT2TokenizerFast

torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Загрузим датасет:

In [2]:
ds = load_dataset("roneneldan/TinyStories")
ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

Посмотрим на несколько примеров:

In [3]:
ds["train"]["text"][:4]

['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leaves that we

Будем использовать готовый токенизатор, например `GPT2TokenizerFast`:

In [4]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

GPT2TokenizerFast(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [5]:
inputs = tokenizer.encode("Once upon a time", return_tensors="pt", add_special_tokens=True)
inputs

tensor([[7454, 2402,  257,  640]])

У этого токенизатора нет выделенного токена для начала последовательности, есть только один специальный токен `<|endoftext|>`. Поэтому, если вы захотите добавить его к началу предложения, придётся делать это вручную:

In [6]:
tokenizer.encode("<|endoftext|>Once upon a time", return_tensors="pt", add_special_tokens=True)

tensor([[50256,  7454,  2402,   257,   640]])

Вместо этого можно добавлять его индекс `50256` в начало всех тензоров уже после токенизации.

Токенизатор можно использовать прямо в загрузчике данных для упаковки токенизированных последовательностей в батчи.

Помимо индексов токенов (`input_ids`) токенизатор вернёт нам `attention_mask` — маску, в которой нулями помечены `pad` токены, использующиеся для выравнивания предложений по длине.

In [7]:
loader = DataLoader(
    dataset=ds["train"]["text"],
    collate_fn=lambda batch: tokenizer(batch, return_tensors="pt", padding=True),
    batch_size=8,
)
batch = next(iter(loader))
batch

{'input_ids': tensor([[ 3198,  1110,    11,  ..., 50256, 50256, 50256],
        [ 7454,  2402,   257,  ..., 50256, 50256, 50256],
        [ 3198,  1110,    11,  ...,   922,  2460,    13],
        ...,
        [ 7454,  2402,   257,  ..., 50256, 50256, 50256],
        [ 7454,  2402,   257,  ..., 50256, 50256, 50256],
        [ 7454,  2402,   257,  ..., 50256, 50256, 50256]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

#### Задание 1. Обучение Transformer Decoder (4 балла)

1. Реализуйте модуль `TransformerDecoder`. Не забудьте сформировать правильную маску внимания! Для этого можете воспользоваться функцией `create_causal_mask` ниже.
2. Создайте с его помощью модель небольшого размера (до 30 млн параметров) и обучите её на `train` части датасета TinyStories. Обучение на всём датасете может занять слишком много времени, так что сделайте столько итераций градиентного спуска, сколько уместится в 10-15 минут обучения. Кроме того, вы можете сделать подвыборку датасета с более короткими текстами, это ускорит обучение модели. Относительно связный текст обычно начинает получаться при снижении cross entropy до 1.5-2.0.
3. После окончания обучения сгенерируйте 10 историй с помощью вашей модели.


In [8]:
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.W_q = nn.Linear(input_dim, hidden_dim)
        self.W_k = nn.Linear(input_dim, hidden_dim)
        self.W_v = nn.Linear(input_dim, input_dim)

    def forward(self, x: Tensor) -> Tensor:
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)
        _, _, D_k = k.shape

        scores = torch.bmm(q, k.permute(0, 2, 1)) / D_k**0.5
        attention_weights = scores.softmax(-1)
        return torch.bmm(attention_weights, v)

class MultiHeadAttention(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, n_heads: int) -> None:
        super().__init__()
        self.heads = nn.ModuleList(
            [ScaledDotProductAttention(input_dim, hidden_dim) for i in range(n_heads)]
        )
        self.W_o = nn.Linear(n_heads * input_dim, input_dim)

    def forward(self, x: Tensor) -> Tensor:
        h = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.W_o(h)

class PositionalEncoding(nn.Module):
    pe: Tensor

    def __init__(self, encoding_size: int, L: int = 10000, max_len: int = 1000):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, encoding_size, 2) * (-math.log(L) / encoding_size)
        )
        pe = torch.zeros(max_len, encoding_size)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, t: Tensor) -> Tensor:
        return self.pe[t]

class TransformerEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, pad_token_id: int = 0, max_position: int = 4096):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=pad_token_id)
        self.position_embeddings = PositionalEncoding(hidden_size, max_len=max_position)
        self.register_buffer(
            "position_ids", torch.arange(max_position).expand((1, -1)), persistent=False
        )

    def forward(self, input_ids: Tensor) -> torch.Tensor:
        B, N = input_ids.size()
        position_ids = self.position_ids[:, : N]
        inputs_embeds = self.token_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)
        embeddings = inputs_embeds + position_embeddings
        return embeddings


In [9]:
class MaskedSelfAttention(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.W_q = nn.Linear(input_dim, hidden_dim)
        self.W_k = nn.Linear(input_dim, hidden_dim)
        self.W_v = nn.Linear(input_dim, input_dim)

    def forward(self, x: Tensor, mask: Tensor) -> Tensor:
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)
        _, _, D_k = k.shape

        attention: Tensor = torch.bmm(q, k.permute(0, 2, 1)) / D_k**0.5
        if mask is not None:
            attention = attention.masked_fill(mask == 0, float("-inf"))
        attention_weights = attention.softmax(-1)
        return torch.bmm(attention_weights, v)


class DecoderLayer(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.self_attention = MaskedSelfAttention(input_dim, hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(inplace=True),
            nn.Linear(input_dim, input_dim),
        )
        self.norm1 = nn.LayerNorm(input_dim)
        self.norm2 = nn.LayerNorm(input_dim)

    def forward(
        self, x: Tensor,
        x_mask: Tensor
    ) -> Tensor:
        h = self.norm1(self.self_attention.forward(
            x=x, mask=x_mask) + x
        )
        return self.norm2(self.mlp(h) + h)


In [10]:
def create_causal_mask(token_ids: Tensor, pad_token_id: int = 0) -> Tensor:
    B, T = token_ids.shape
    # маска для <pad> токенов
    pad_mask = (token_ids != pad_token_id).unsqueeze(1)  # B x 1 x T
    # маска нижнетреугольной матрицы
    causal_mask = torch.tril(
        torch.ones((T, T), device=token_ids.device)
    ).bool()  # T x T
    return pad_mask & causal_mask  # B x T x T


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, n_layers: int):
        super().__init__()
        self.embeds = TransformerEmbeddings(vocab_size, hidden_size)
        self.trunc = nn.ModuleList([DecoderLayer(hidden_size, hidden_size) for i in range(n_layers)])
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids: Tensor, attention_mask: Tensor) -> Tensor:
        h = self.embeds.forward(input_ids)

        for layer in self.trunc:
            h = layer.forward(h, attention_mask)

        logits = self.head(h)
        return logits

In [11]:
train_texts = ds["train"]["text"][:15000]
loader = DataLoader(
    dataset=train_texts,
    collate_fn=lambda batch: tokenizer(batch, return_tensors="pt", padding=True),
    batch_size=12,
)

vocab_size = tokenizer.vocab_size
model = TransformerDecoder(vocab_size=vocab_size, hidden_size=256, n_layers=8)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4)

model.train()
for epoch in range(7):
    for i, batch in enumerate(loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        targets = input_ids[:, 1:]

        causal_mask = create_causal_mask(input_ids[:, :-1])
        training_mask = causal_mask & attention_mask[:, :-1].unsqueeze(1)

        logits = model(input_ids[:, :-1], training_mask)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            ignore_index=tokenizer.pad_token_id
        )
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if i % 100 == 0:
            print(f"epoch {epoch}, batch {i}, loss: {loss.item():.4f}")


Total parameters: 28,421,713
epoch 0, batch 0, loss: 10.8841
epoch 0, batch 100, loss: 5.2160


Token indices sequence length is longer than the specified maximum sequence length for this model (1106 > 1024). Running this sequence through the model will result in indexing errors


epoch 0, batch 200, loss: 4.2280
epoch 0, batch 300, loss: 4.2439
epoch 0, batch 400, loss: 4.2770
epoch 0, batch 500, loss: 3.6385
epoch 0, batch 600, loss: 3.4254
epoch 0, batch 700, loss: 3.6662
epoch 0, batch 800, loss: 3.1933
epoch 0, batch 900, loss: 3.5810
epoch 0, batch 1000, loss: 3.6819
epoch 0, batch 1100, loss: 3.0962
epoch 0, batch 1200, loss: 3.0798
epoch 1, batch 0, loss: 3.3649
epoch 1, batch 100, loss: 3.5575
epoch 1, batch 200, loss: 2.8805
epoch 1, batch 300, loss: 3.3044
epoch 1, batch 400, loss: 3.4665
epoch 1, batch 500, loss: 2.8760
epoch 1, batch 600, loss: 2.7252
epoch 1, batch 700, loss: 3.0540
epoch 1, batch 800, loss: 2.6256
epoch 1, batch 900, loss: 3.0973
epoch 1, batch 1000, loss: 3.2580
epoch 1, batch 1100, loss: 2.5890
epoch 1, batch 1200, loss: 2.6225
epoch 2, batch 0, loss: 2.9154
epoch 2, batch 100, loss: 3.1541
epoch 2, batch 200, loss: 2.5044
epoch 2, batch 300, loss: 2.9588
epoch 2, batch 400, loss: 3.1363
epoch 2, batch 500, loss: 2.5352
epoch 2,

Для генерации примеров можно использовать простейшую стратегию с семплированием следующего токена:

In [12]:
def generate(
    model: TransformerDecoder, input_ids: Tensor, max_new_tokens: int = 200
) -> Tensor:
    for t in range(max_new_tokens):
        mask = create_causal_mask(input_ids)
        logits = model.forward(input_ids, mask)[:, -1]
        # new_token = logits.argmax(dim=-1, keepdim=True)
        new_token = torch.multinomial(logits.softmax(-1), num_samples=1)
        input_ids = torch.cat([input_ids, new_token], dim=1)

    return input_ids

In [15]:
prompt = "Once upon a time"
for i in range(10):
  print("######################### text №", i + 1)
  inputs = tokenizer(prompt, return_tensors="pt").to(device="cuda")
  generate_ids = generate(model, inputs.input_ids, max_new_tokens=200)
  response = tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
  print(response)
  print()

######################### text № 1
Once upon a time, there was a little girl named Lily. She loved to drink cocoa at cocoa. One weekend was a sweet and she never saw a wild cake in the water. Her mom told her that she could enjoy what it was.

Lily was scared, so she asked her mom if she could touch it. Her mom said yes and gave her some of cocoa. Lily smiled and wiped as she was about to find food. They decided to go for a walk in the street and went to the different cocoa.

The next day, Lily was eating cell of cocoa for all the fish. She was even ready. Finally, her mom held it tightly in a pink spot for her mom and said they lived happily ever after. The mom was proud of Lily and for finding what she gave. Lily felt so brave and loved her mom much. They laughed and smiled. They had a fun time together in the right bed. They enjoyed talking about how it was to all.

And never

######################### text № 2
Once upon a time there was a daughter. She was very brave outside to exp